In [1]:
import os
import gc
import pandas as pd
import numpy as np
from typing import Dict, Union

# Helper function to map month to season
def get_season(month: int) -> str:
    """Maps month (1-12) to Northern Hemisphere season."""
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return 'Unknown' # Should not happen with valid months

def add_temporal_features(df: pd.DataFrame, inplace: bool = False) -> Union[pd.DataFrame, None]:
    """
    Adds temporal features to a DataFrame based on its 'time' index level.

    Handles DataFrames with potentially non-unique MultiIndex.

    The DataFrame must have a MultiIndex with the second level named 'time'
    and containing datetime-like objects.

    Features added:
    - year: The year.
    - month: The month (1-12).
    - day: The day of the month (1-31).
    - hour: The hour of the day (0-23) - relevant for hourly data.
    - dayofweek: The day of the week (0=Monday, 6=Sunday).
    - dayofyear: The day of the year (1-366).
    - weekofyear: The ISO week number (1-53).
    - quarter: The quarter (1-4).
    - season: The season (Winter, Spring, Summer, Autumn).
    - is_weekend: Binary flag (1 if Saturday/Sunday, 0 otherwise).
    - time_epoch: Seconds since the epoch.

    Args:
        df (pd.DataFrame): Input DataFrame with a 'time' index level.
        inplace (bool): If True, modify the DataFrame in place.
                        If False (default), return a modified copy.

    Returns:
        pd.DataFrame or None: The DataFrame with added temporal features,
                              or None if inplace=True.

    Raises:
        ValueError: If the DataFrame does not have a MultiIndex or
                    if the second index level is not named 'time'.
        TypeError: If the 'time' index level is not datetime-like.
    """
    if not isinstance(df.index, pd.MultiIndex):
        raise ValueError("Input DataFrame must have a MultiIndex.")
    # Get index names safely, handle potential None case
    idx_names = df.index.names
    if idx_names is None or 'time' not in idx_names or idx_names.index('time') != 1:
         # Check if 'time' exists and is the second level (index 1)
        raise ValueError("The second level of the MultiIndex must be named 'time'.")

    # Work on a copy if not inplace
    output_df = df if inplace else df.copy()

    # Ensure the time index level is datetime
    time_index = output_df.index.get_level_values('time')
    if not pd.api.types.is_datetime64_any_dtype(time_index):
        try:
            # Attempt conversion if necessary
            output_df.index = output_df.index.set_levels(pd.to_datetime(time_index), level='time')
            time_index = output_df.index.get_level_values('time') # Refresh time_index after conversion
            if not pd.api.types.is_datetime64_any_dtype(time_index):
                 raise TypeError("Could not convert 'time' index level to datetime.")
        except Exception as e:
             raise TypeError(f"The 'time' index level is not datetime-like and conversion failed: {e}")

    # --- Extract features and assign using .to_numpy() ---
    output_df['year'] = time_index.year.to_numpy()
    output_df['month'] = time_index.month.to_numpy()
    output_df['day'] = time_index.day.to_numpy()
    output_df['hour'] = time_index.hour.to_numpy() # Will be 0 for daily/weekly
    output_df['dayofweek'] = time_index.dayofweek.to_numpy() # Monday=0, Sunday=6
    output_df['dayofyear'] = time_index.dayofyear.to_numpy()
    # Use isocalendar().week for ISO 8601 week number
    output_df['weekofyear'] = time_index.isocalendar().week.astype(int).to_numpy()
    output_df['quarter'] = time_index.quarter.to_numpy()
    # Apply season based on the 'month' column we just created
    output_df['season'] = output_df['month'].apply(get_season).to_numpy()
    # Calculate 'is_weekend' based on the 'dayofweek' column
    output_df['is_weekend'] = (output_df['dayofweek'] >= 5).astype(int).to_numpy()
    # Add epoch time (might be useful for some models)
    #output_df['time_epoch'] = (time_index.astype(np.int64) // 10**9).to_numpy()

    if not inplace:
        return output_df
    else:
        return None


def add_severity_feature(df: pd.DataFrame, num_bins: int = 10, inplace: bool = False) -> Union[pd.DataFrame, None]:
    """
    Adds a 'severity' column based on quantiles of 'customers_out'.

    Bins the 'customers_out' column into a specified number of quantiles.
    Handles potential duplicate bin edges by merging bins.

    Args:
        df (pd.DataFrame): Input DataFrame with a 'customers_out' column.
        num_bins (int): The number of desired severity classes (quantiles).
        inplace (bool): If True, modify the DataFrame in place.
                        If False (default), return a modified copy.

    Returns:
        pd.DataFrame or None: The DataFrame with the added 'severity' column,
                              or None if inplace=True.

    Raises:
        ValueError: If the 'customers_out' column is missing.
    """
    if 'customers_out' not in df.columns:
        raise ValueError("DataFrame must contain a 'customers_out' column.")

    if not inplace:
        df = df.copy()

    # Use qcut to create bins based on quantiles
    # labels=False returns integer indicators (0 to num_bins-1)
    # duplicates='drop' handles cases where quantile boundaries are not unique
    # (e.g., many zeros), merging bins as necessary.
    try:
        severity_bins, bin_edges = pd.qcut(
            df['customers_out'],
            q=num_bins,
            labels=False,
            duplicates='drop',
            retbins=True # Get the bin edges for potential inspection
        )
        df['severity'] = severity_bins.astype(float) # Keep as float to allow NaNs
         # You might want to convert to Int64 later if NaNs are filled
         # df['severity'] = severity_bins.astype(pd.Int64Dtype())


        print(f"Created 'severity' column with {severity_bins.nunique()} unique bins.")
        # Optional: Print bin edges for reference (can be verbose)
        # print("Bin edges for severity:")
        # print(bin_edges)

    except ValueError as e:
        # Handle cases where qcut fails (e.g., not enough unique non-null values)
        print(f"Warning: Could not create {num_bins} severity bins: {e}")
        print("Assigning NaN to 'severity' column.")
        df['severity'] = np.nan


    if not inplace:
        return df
    else:
        return None

In [2]:
dataframe_names = [
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_weekly_sum_weather.parquet',
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_weekly_mean_weather.parquet',
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_daily_sum_weather.parquet',
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_daily_mean_weather.parquet',
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_hourly_sum_weather.parquet',
    '/kaggle/input/dynamic-rhythms-weather-added/combined_county_hourly_mean_weather.parquet',
]

In [3]:
for dataframe_name in dataframe_names:
    print(f"Processing {dataframe_name.replace('/kaggle/input/dynamic-rhythms-weather-added/', '')}")
    df = pd.read_parquet(dataframe_name)
    df_temporal = add_temporal_features(df)
    del df
    gc.collect()
    df_full = add_severity_feature(df_temporal, num_bins=10)
    del df_temporal
    gc.collect()
    #print("\n--- Daily DataFrame with Temporal and Severity Features Head ---")
    #print(df_full[['customers_out', 'severity']].head(5))

    # Generate processed filename
    processed_name = dataframe_name.replace('/kaggle/input/dynamic-rhythms-weather-added/', '').replace('.parquet', '_processed.parquet')
    gc.collect()
    # Save the processed DataFrame to Parquet
    df_full.to_parquet(processed_name)
    print(f"Processed DataFrame saved to: {processed_name}")
    gc.collect()
    del df_full

Processing combined_county_weekly_sum_weather.parquet
Created 'severity' column with 10 unique bins.
Processed DataFrame saved to: combined_county_weekly_sum_weather_processed.parquet
Processing combined_county_weekly_mean_weather.parquet
Created 'severity' column with 10 unique bins.
Processed DataFrame saved to: combined_county_weekly_mean_weather_processed.parquet
Processing combined_county_daily_sum_weather.parquet
Created 'severity' column with 10 unique bins.
Processed DataFrame saved to: combined_county_daily_sum_weather_processed.parquet
Processing combined_county_daily_mean_weather.parquet
Created 'severity' column with 10 unique bins.
Processed DataFrame saved to: combined_county_daily_mean_weather_processed.parquet
Processing combined_county_hourly_sum_weather.parquet
Created 'severity' column with 10 unique bins.
Processed DataFrame saved to: combined_county_hourly_sum_weather_processed.parquet
Processing combined_county_hourly_mean_weather.parquet
Created 'severity' column

In [6]:
print("Finished")

Finished


In [4]:
#pd.set_option('display.max_columns', None)

In [5]:
#pd.read_parquet("/kaggle/working/combined_county_weekly_mean_weather_processed.parquet").head()